# YOLOv8 vs Faster R-CNN: vehicle detection benchmark

Compares the detectors used in the Smart Traffic Management app on the **vehicle classes of COCO val2017** (car, motorcycle, bus, truck):

| Measure | How |
|---|---|
| Accuracy | mAP@0.5 and mAP@0.5:0.95 with the official COCO evaluation (`pycocotools`), plus AP per vehicle type |
| Speed | inference time per image and frames per second, after 10 warm-up images |
| Low light | the same images darkened artificially, without and with the app's low-light enhancement |

**Kaggle settings (right panel):** Accelerator → **GPU T4 x2** (one GPU is used), Internet → **On**.
Run time: about 20–30 minutes for 500 images.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q ultralytics pycocotools

## 1. Get the code and the COCO validation images
The 5,000 COCO val2017 images (about 1 GB) and the annotations (about 250 MB) are downloaded from the official COCO site.

In [ ]:
%cd /kaggle/working
!git clone -q https://github.com/pavanadityav-ai/smart-traffic-management.git
!mkdir -p coco
!wget -q http://images.cocodataset.org/zips/val2017.zip && unzip -q val2017.zip -d coco && rm val2017.zip
!wget -q http://images.cocodataset.org/annotations/annotations_trainval2017.zip && unzip -q annotations_trainval2017.zip -d coco && rm annotations_trainval2017.zip
!ls coco coco/annotations | head

## 2. Run the benchmark
Models: YOLOv8 n, s and m, and Faster R-CNN (ResNet-50 FPN v2). All are the standard COCO-pretrained weights, so the comparison is fair: same images, same classes, same evaluation code.

In [ ]:
%cd /kaggle/working/smart-traffic-management
!python benchmark/compare_models.py --coco-dir /kaggle/working/coco --max-images 500 --out /kaggle/working/results

## 3. Results

In [ ]:
import json
import pandas as pd
from IPython.display import Image, display

report = json.load(open("/kaggle/working/results/benchmark.json"))
print(report["dataset"], "|", report["images"], "images |", report["device"])
table = pd.DataFrame(report["models"]).drop(columns="ap_per_class").set_index("name")
display(table)
display(pd.DataFrame({m["name"]: m["ap_per_class"] for m in report["models"]}).T.rename_axis("AP@0.5:0.95"))
display(pd.DataFrame(report["low_light"]).set_index("model"))
display(Image("/kaggle/working/results/benchmark.png"))

## Observations
*(Write 3–4 points after the run: which model is most accurate, which is fastest, which one you would use live and why, and what the low-light test shows.)*

## Next step
Download `benchmark.json` and `benchmark.png` from the **Output** tab (`results/` folder), and add them to the `results/` folder of the GitHub repository. The app's **Model comparison** page then shows the charts.